In [ ]:
#!/usr/bin/env python3
"""
NB09: Maximal Covering Location Problem (MCLP)
Replaces the KMeans heuristic with an exact Integer Linear Programming (ILP) solver.
- Demands: Sampled mission locations.
- Candidates: Extracted from a grid across the region.
- Objective: Maximize coverage (demand within 500m) given a budget constraint on p (number of new AEDs).
"""



# NB 09 — ILP/MCLP Exact Optimization

Replaces the KMeans clustering heuristic with **Integer Linear Programming (ILP)**
solving the Maximal Covering Location Problem (MCLP).
We aim to maximize the demand covered within a 500m radius given exactly p new devices.

**Expected runtime: ~5-15 minutes**


In [ ]:
Setup
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import pulp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Load config
PROJECT_ROOT = Path('.')
V3_DIR   = PROJECT_ROOT / 'mda_project' / 'data' / 'processed_v3'
OUT_DIR  = PROJECT_ROOT / 'mda_project' / 'data' / 'output'
FIG_DIR  = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

EARTH_R = 6371.0088

def haversine_dist(lat1, lon1, lat2, lon2):
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * EARTH_R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))




In [ ]:
Load Data
print("Loading data...")
mission = pd.read_parquet(V3_DIR / 'mission_records_v3.parquet')
aed = pd.read_parquet(V3_DIR / 'aed_records_v3.parquet')

# Clean missions
m = mission.dropna(subset=['latitude', 'longitude']).copy()
# Only keep missions not currently covered within 500m
m = m[m['dist_to_aed_km'] > 0.5]
print(f"Total uncovered missions: {len(m)}")

# For ILP tractability, sample 5k uncovered missions
N_DEMAND = 5000
if len(m) > N_DEMAND:
    # Stratified/weighted sample based on density could be better, but simple random sample works as a proxy
    demand_df = m.sample(n=N_DEMAND, random_state=SEED).reset_index(drop=True)
else:
    demand_df = m.reset_index(drop=True)

demand_points = demand_df[['latitude', 'longitude']].values




In [ ]:
Generate Candidate Sites
# Create a grid across Belgium. Only keep grid points that are >500m from existing AEDs
# and within 1km of at least one demand point.
lat_min, lat_max = 49.5, 51.5
lon_min, lon_max = 2.5, 6.2
grid_lat = np.linspace(lat_min, lat_max, 60)
grid_lon = np.linspace(lon_min, lon_max, 60)
ll_lat, ll_lon = np.meshgrid(grid_lat, grid_lon)
candidates_raw = np.column_stack([ll_lat.ravel(), ll_lon.ravel()])

print(f"Raw candidate grid size: {len(candidates_raw)}")

# Filter using haversine matrix
from scipy.spatial.distance import cdist

def haversine_matrix(pts1, pts2):
    # pts: (N, 2) array of [lat, lon]
    # returns (N, M) distance matrix
    lat1 = np.radians(pts1[:, 0:1])
    lon1 = np.radians(pts1[:, 1:2])
    lat2 = np.radians(pts2[:, 0:1]).T
    lon2 = np.radians(pts2[:, 1:2]).T
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    # ensure a is in [0, 1] due to float precision
    a = np.clip(a, 0, 1)
    
    return 2 * EARTH_R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

# Nearest AED dist for candidates
cand_to_aed_dist = np.min(haversine_matrix(candidates_raw, aed[['latitude', 'longitude']].values), axis=1)
candidates_filt = candidates_raw[cand_to_aed_dist > 0.4] # Must be at least 400m from existing AED

# Must be near demand
cand_to_dem_dist = np.min(haversine_matrix(candidates_filt, demand_points), axis=1)
candidates = candidates_filt[cand_to_dem_dist <= 1.0] # Within 1km of a demand
print(f"Filtered candidate sites: {len(candidates)}")




In [ ]:
Optimization Setup (MCLP)
# Demand points i in I
# Candidate sites j in J
# Coverage distance D = 0.5 km
# y_i: binary, 1 if demand i is covered
# x_j: binary, 1 if AED is placed at candidate j
# p: number of AEDs

COVER_THRESH = 0.5

print("Calculating distance matrix for ILP...")
D_ij = haversine_matrix(demand_points, candidates)

# a_ij matrix: 1 if distance <= 0.5, 0 otherwise
A = (D_ij <= COVER_THRESH).astype(int)

# Set of candidates covering each demand i
N_i = [np.where(A[i, :] == 1)[0] for i in range(len(demand_points))]

print("Setting up PuLP models...")
p_levels = [10, 30, 50, 70, 100]
results = []
opt_locations = {}

for p in p_levels:
    print(f"Solving for p={p}...")
    prob = pulp.LpProblem("MCLP", pulp.LpMaximize)
    
    # Variables
    y = pulp.LpVariable.dicts("y", range(len(demand_points)), cat='Binary')
    x = pulp.LpVariable.dicts("x", range(len(candidates)), cat='Binary')
    
    # Objective
    prob += pulp.lpSum([y[i] for i in range(len(demand_points))])
    
    # Constraints
    # 1. Total AEDs cannot exceed p
    prob += pulp.lpSum([x[j] for j in range(len(candidates))]) == p
    
    # 2. Demand i is covered only if at least one selected facility j covers it
    for i in range(len(demand_points)):
        prob += pulp.lpSum([x[j] for j in N_i[i]]) >= y[i]
        
    # Solve
    prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=60))
    
    # Extract results
    status = pulp.LpStatus[prob.status]
    covered_count = sum(y[i].varValue for i in range(len(demand_points)))
    covered_pct = covered_count / len(demand_points)
    
    selected_j = [j for j in range(len(candidates)) if x[j].varValue > 0.5]
    opt_locations[f'P{p}'] = candidates[selected_j]
    
    results.append({
        'New_AEDs': p,
        'Solver_Status': status,
        'Demands_Covered': int(covered_count),
        'Covered_Pct': covered_pct * 100
    })

res_df = pd.DataFrame(results)
print(res_df.to_string())
res_df.to_csv(FIG_DIR / 'table_09_mclp_results.csv', index=False)




In [ ]:
Figure: ILP vs Current
print("Rendering Fig 6 (MCLP Expansion)...")
fig6, ax = plt.subplots(figsize=(10, 8), dpi=200)

boundary = gpd.read_file('mda_project/data/raw/BELGIUM_-_Provinces.geojson')
bel = boundary.to_crs(4326)
bel.plot(ax=ax, color='#f0efeb', edgecolor='#333333', lw=0.8, zorder=1)

# Plot demand points (sampled)
ax.scatter(demand_points[:, 1], demand_points[:, 0], s=2, color='gray', alpha=0.3, label='Uncovered Demand', zorder=2)

# Plot selected P=100 sites
sel = opt_locations['P100']
ax.scatter(sel[:, 1], sel[:, 0], s=30, color='#e74c3c', edgecolors='white', marker='*', label='Optimal P=100 Placements', zorder=4)

ax.set_title('(e) Exact Optimization (MCLP P=100)\nDemand nodes sampled at N=5000', fontsize=14, fontweight='bold', pad=10)
ax.legend(loc='lower left', framealpha=0.9)
ax.set_axis_off()

fig6.tight_layout()
fig6.savefig(FIG_DIR / 'fig6_mclp_map.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.close(fig6)
print("  -> saved fig6_mclp_map.png")
print("\n=== NB09 Complete ===")

